In [25]:
# ==========================================
# STEP 1: INSTALL PACKAGES (Google Colab / local Jupyter)
# ==========================================
print("Installing model development libraries...")

# Only what the pipeline actually needs. The old list (diffusers, accelerate,
# safetensors, llama-cpp-python, sentencepiece) was unused -- and
# llama-cpp-python compiles from source, which is slow/fragile on Colab.
# transformers will pull in its own dependencies (torch, safetensors, ...).
!pip install -q -U transformers gguf

print("\n✅ Step 1 complete! Core packages installed.")

Installing model development libraries...

✅ Step 1 complete! Core packages installed.


In [4]:
# ==========================================
# STEP 2: CREATE THE .NET C# Q&A TRAINING CORPUS
# ==========================================
print("Generating comprehensive .NET C# Q&A data asset with code scripts...")

# Raw string literal so the C# braces/backslashes are kept literally.
# NOTE: the old corpus had a stray backslash (`$"Error: ...`); fixed to valid C#.
dot_net_massive_corpus = r"""
Question: How do you implement a standard Web API Controller with Dependency Injection in .NET?
Answer: You create a class deriving from ControllerBase, decorate it with the ApiController attribute, and accept your dependencies natively via the class constructor parameters.
```csharp
[ApiController]
[Route("api/[controller]")]
public class UsersController : ControllerBase
{
    private readonly IUserService _userService;
    public UsersController(IUserService userService)
    {
        _userService = userService;
    }
    [HttpGet("{id}")]
    public async Task GetUser(int id)
    {
        var user = await _userService.GetByIdAsync(id);
        if (user == null) return NotFound();
        return Ok(user);
    }
}
```

Question: How do you register services in the .NET Dependency Injection container?
Answer: You configure services inside the Program.cs file using the WebApplicationBuilder's Services collection, choosing between Transient, Scoped, or Singleton lifetimes.
```csharp
var builder = WebApplication.CreateBuilder(args);
builder.Services.AddTransient();
builder.Services.AddScoped();
builder.Services.AddSingleton();
var app = builder.Build();
app.Run();
```

Question: How do you write an asynchronous method using Async and Await in C#?
Answer: You decorate the method signature with the async keyword, return a Task or Task, and use the await operator to pause execution non-blockingly until the underlying operation completes.
```csharp
public async Task DownloadDataAsync(string url)
{
    using var client = new HttpClient();
    try
    {
        string result = await client.GetStringAsync(url);
        return result;
    }
    catch (Exception ex)
    {
        return $"Error: {ex.Message}";
    }
}
```

Question: How do you write a LINQ query to filter and project data from a collection in C#?
Answer: You can use either method syntax with lambda expressions or query syntax to filter elements matching a condition and project them into new shapes or anonymous types.
```csharp
public List GetActiveUserNames(List users)
{
    var activeUsers = users
        .Where(u => u.IsActive == true && u.Age > 18)
        .Select(u => new UserDto { Id = u.Id, FullName = u.Name })
        .ToList();
    return activeUsers;
}
```

Question: How do you configure a database context class for Entity Framework Core?
Answer: You create a class that inherits from DbContext, expose DbSet properties for your database tables, and override the OnConfiguring or OnModelCreating methods for mapping rules.
```csharp
public class ApplicationDbContext : DbContext
{
    public ApplicationDbContext(DbContextOptions options) : base(options) {}
    public DbSet Products { get; set; }
    protected override void OnModelCreating(ModelBuilder modelBuilder)
    {
        modelBuilder.Entity().Property(p => p.Price).HasPrecision(18, 2);
    }
}
```
"""

# Multiply the text data blocks to create a deep training array for our model
dataset_multiplier = 40
massive_text_data = dot_net_massive_corpus.strip() * dataset_multiplier

# Save directly to the local (Colab) folder
with open("input.txt", "w", encoding="utf-8") as f:
    f.write(massive_text_data)

# Read total lines count to confirm size
with open("input.txt", "r", encoding="utf-8") as f:
    lines = f.readlines()

print(f"✅ Success! Generated Dataset File contains: {len(lines)} lines.")

# Informational only: character-level unique count (the model uses WORD tokens, see Step 3)
chars = sorted(list(set(massive_text_data)))
print(f"Unique characters (informational only): {len(chars)}")


Generating comprehensive .NET C# Q&A data asset with code scripts...
✅ Success! Generated Dataset File contains: 3041 lines.
Unique characters (informational only): 73


In [5]:
# ==========================================
# STEP 3: WORD-LEVEL TOKENIZER SETUP
# ==========================================
import torch
import re

print("Parsing dataset into word-level tokens...")

# Read the local data file generated from Step 2
with open("input.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

# Use regular expressions to split text cleanly into individual words, symbols, and punctuation
token_pattern = r"\w+|[^\w\s]|\n"
all_tokens = re.findall(token_pattern, raw_text)

# Establish a clean, unique word dictionary vocabulary
vocab = sorted(list(set(all_tokens)))
vocab_size = len(vocab)

# Create structural word-to-integer mappings
word_to_int = {word: i for i, word in enumerate(vocab)}
int_to_word = {i: word for i, word in enumerate(vocab)}

encode = lambda s: [word_to_int[w] for w in re.findall(token_pattern, s) if w in word_to_int]
def decode(indices):
    # Reconstruct tokens back into natural readable text layout
    text_out = ""
    for idx in indices:
        word = int_to_word[idx]
        if word == "\n":
            text_out += "\n"
        elif word in [".", ",", "(", ")", "[", "]", "{", "}", ";", ":", "=>"]:
            text_out += word
        else:
            text_out += " " + word
    return text_out.strip()

# Convert entire dataset into structural token vectors
data = torch.tensor(encode(raw_text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

# Update sequence block settings for word limits
batch_size = 16
# Long enough to hold a full Q&A pair (~140 words) PLUS the next question header.
# 64 was too short: the model only ever saw local fragments, which is why
# generation blended one answer into another instead of reproducing them cleanly.
block_size = 256
device = 'cuda' if torch.cuda.is_available() else 'cpu'

def get_batch(split='train'):
    data_src = train_data if split == 'train' else val_data
    ix = torch.randint(len(data_src) - block_size, (batch_size,))
    x = torch.stack([data_src[i:i+block_size] for i in ix])
    y = torch.stack([data_src[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

print(f"✅ Word Tokenization Complete! Total Dataset Tokens: {len(data)}")
print(f"Dictionary Size (Unique Words & Symbols): {vocab_size}")


Parsing dataset into word-level tokens...
✅ Word Tokenization Complete! Total Dataset Tokens: 26640
Dictionary Size (Unique Words & Symbols): 216


In [6]:
# ==========================================
# STEP 4: MODEL CONFIGURATION
# ==========================================
from transformers import LlamaConfig, LlamaForCausalLM

print("Building standard LLaMA structural nodes matching word-level dimensions...")

config = LlamaConfig(
    vocab_size=vocab_size,
    hidden_size=256,
    intermediate_size=2048,      # Expanded memory capacity for C# rules
    num_hidden_layers=4,
    num_attention_heads=4,
    num_key_value_heads=4,
    max_position_embeddings=block_size,  # must stay == GGUF context_length (Step 7)
    bos_token_id=0,
    eos_token_id=1,
    pad_token_id=2
)

model = LlamaForCausalLM(config).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4) # Slightly lower learning rate for stability

print("✅ Model layout anchored and ready.")


Building standard LLaMA structural nodes matching word-level dimensions...
✅ Model layout anchored and ready.


In [18]:
# ==========================================
# STEP 5: THE CORE TRAINING LOOP
# ==========================================
print("Beginning training passes... optimizing weights against C# structural layout...")
model.train()

# More steps + longer context (Step 3) => the model memorizes each Q&A cleanly.
# 1200 steps was under-trained (only ~0.75 passes over the data), so positions the
# model had not memorized yet produced blended/garbage text during generation.
max_iters = 3000
eval_interval = 500

@torch.no_grad()
def estimate_loss():
    """Average loss over a few random batches for train/val, to confirm memorization."""
    model.eval()
    out = {}
    for split in ["train", "val"]:
        losses = torch.zeros(50)
        for k in range(50):
            xb, yb = get_batch(split)
            losses[k] = model(xb, labels=yb).loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

for step in range(max_iters):
    xb, yb = get_batch("train")

    # Calculate output logits and loss metrics
    outputs = model(input_ids=xb, labels=yb)
    loss = outputs.loss

    # Perform standard backward adjustments parameters pass
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if step % eval_interval == 0 or step == max_iters - 1:
        losses = estimate_loss()
        print(f"Step {step:04d} | Train Loss: {losses['train']:.4f} | Val Loss: {losses['val']:.4f}")

print(f"\n✅ Optimization complete! Final structural loss reached: {loss.item():.4f}")

Beginning training passes... optimizing weights against C# structural layout...
Step 0000 | Train Loss: 0.0111 | Val Loss: 0.0117
Step 0500 | Train Loss: 0.0105 | Val Loss: 0.0105
Step 1000 | Train Loss: 0.0109 | Val Loss: 0.0105
Step 1500 | Train Loss: 0.0740 | Val Loss: 0.0741
Step 2000 | Train Loss: 0.0109 | Val Loss: 0.0107
Step 2500 | Train Loss: 0.0107 | Val Loss: 0.0101
Step 2999 | Train Loss: 0.0102 | Val Loss: 0.0102

✅ Optimization complete! Final structural loss reached: 0.0107


In [19]:
# ==========================================
# STEP 6: SAVE HUGGING FACE TARGET DIRECTORY
# ==========================================
output_directory = "./massive_net_llama"
model.save_pretrained(output_directory)

# Save the ACTUAL word-level tokenizer vocabulary (one token per line).
# BUGFIX: the old code wrote the character-level set `chars`, which did NOT match
# the model's word-level embedding table -- it would break any downstream loader.
with open(f"{output_directory}/vocab.txt", "w", encoding="utf-8") as f:
    for word in vocab:
        f.write(word + "\n")

print(f"✅ Model matrices written securely to local directory path: {output_directory}")
print(f"Word-level vocab ({len(vocab)} tokens) saved to vocab.txt")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model matrices written securely to local directory path: ./massive_net_llama
Word-level vocab (216 tokens) saved to vocab.txt


In [21]:
# ==========================================
# STEP 7: DIRECT LOCAL TENSOR-TO-GGUF WRITER
# ==========================================
print("\U0001f4e6 Compiling PyTorch matrices into pure GGUF format locally...")

from gguf import GGUFWriter
from transformers import AutoModelForCausalLM

model_dir = "./massive_net_llama"
output_gguf = "./dotnet_code_model.gguf"

# 1. Initialize a clean GGUF binary schema writer stream mapping
writer = GGUFWriter(output_gguf, arch="llama")

# 2. Bind core dimensions into GGUF metadata headers manually.
#    CRITICAL: every value below must match the LlamaConfig from Step 4.
writer.add_name("TinyDotNetLLM")
writer.add_context_length(block_size)  # BUGFIX: was hardcoded 128, did not match max_position_embeddings
writer.add_embedding_length(256)
writer.add_block_count(4)

# 3. The tokenizer here is a CUSTOM WORD-LEVEL tokenizer (Step 3), not llama.cpp's
#    built-in BPE/sentencepiece tokenizer. Mark it "no_vocab" so llama.cpp loads the
#    weights and expects you to attach the matching vocabulary (the word-level
#    vocab.txt written in Step 6) as an external tokenizer.
writer.add_tokenizer_model("no_vocab")
writer.add_bos_token_id(0)
writer.add_eos_token_id(1)
writer.add_pad_token_id(2)

# 4. Pull our saved model parameters from your local notebook folder
hf_model = AutoModelForCausalLM.from_pretrained(model_dir)
state_dict = hf_model.state_dict()

print(f"Processing and converting {len(state_dict)} individual tensor layer arrays...")

# 5. Map and register every single tensor array into the writer buffer
for tensor_name, tensor_value in state_dict.items():
    # Convert weights to standard Float32 precision arrays for the GGUF spec engine
    numpy_data = tensor_value.detach().cpu().numpy().astype("float32")
    writer.add_tensor(tensor_name, numpy_data)

# 6. EXECUTE THE COMPLETE STEP-BY-STEP WRITE LIFECYCLE
print("Writing structural binary nodes out to disk space...")
writer.write_header_to_file()   # Writes magic bytes and layout counters
writer.write_kv_data_to_file()  # Closes the Key-Value metadata channel cleanly
writer.write_tensors_to_file()  # Safely dumps multi-dimensional matrix values
writer.close()                  # Locks file changes

print(f"\U0001f389 Clean Execution! Your fully offline GGUF file has been built at: '{output_gguf}'")
print("Note: this model uses a custom word-level tokenizer. Use vocab.txt from Step 6 as the tokenizer vocabulary (llama.cpp tokenizer.ggml.model = no_vocab).")

📦 Compiling PyTorch matrices into pure GGUF format locally...


Loading weights:   0%|          | 0/39 [00:00<?, ?it/s]

Processing and converting 39 individual tensor layer arrays...
Writing structural binary nodes out to disk space...
🎉 Clean Execution! Your fully offline GGUF file has been built at: './dotnet_code_model.gguf'
Note: this model uses a custom word-level tokenizer. Use vocab.txt from Step 6 as the tokenizer vocabulary (llama.cpp tokenizer.ggml.model = no_vocab).


In [26]:
# ==========================================
# STEP 8: WORD-LEVEL INFERENCE DEMO
# ==========================================
print("Loading model matrices from local memory space...")

import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM

# Load the trained model layers
native_llm = AutoModelForCausalLM.from_pretrained("./massive_net_llama").to(device)
native_llm.eval()

# Use the FULL question, exactly as it appears in the training corpus.
# BUGFIX: the old prompt was cut off mid-sentence ("...Dependency Injection"
# without "container?"), which was off-distribution and contributed to garbage.
test_prompt = "Question: How do you register services in the .NET Dependency Injection container?"
print(f"\n--- Prompting Model with: '{test_prompt}' ---")

# Encode prompt using our word tokenizer
input_indices = encode(test_prompt)
input_tensor = torch.tensor([input_indices], dtype=torch.long).to(device)

# Greedy decoding (temperature=0) faithfully reproduces the memorized corpus.
# For sampling instead, set e.g. temperature = 0.7 and torch.manual_seed(1337).
temperature = 0.0

with torch.no_grad():
    new_ids = []
    for _ in range(150):  # Generate up to 150 words
        cond_tensor = input_tensor[:, -block_size:]

        outputs = native_llm(cond_tensor)
        logits = outputs.logits[:, -1, :]

        if temperature <= 0:
            next_token = int(logits.argmax(dim=-1).item())  # greedy: most likely token
        else:
            probabilities = F.softmax(logits / temperature, dim=-1)
            next_token = torch.multinomial(probabilities, num_samples=1).item()

        new_ids.append(next_token)
        input_tensor = torch.cat((input_tensor, torch.tensor([[next_token]], device=device)), dim=1)

        # Stop cleanly once a full answer has been produced:
        #  - a complete ``` ``` code fence is 6 backtick tokens (open+close), or
        #  - the model has moved on to the next example's "Question:" header.
        # (Counting decoded text for ``` did NOT work -- decode() puts spaces
        #  between backtick tokens, so the old stop check never fired.)
        backtick_count = sum(1 for t in new_ids if int_to_word[t] == "`")
        if backtick_count >= 6:
            break
        if "Question:" in decode(new_ids):
            break

    generated_indices = input_indices + new_ids

print("\n\U0001f916 Model Output:")
print(decode(generated_indices))

# Ground truth straight from the training corpus, for comparison
def get_ground_truth(question):
    i = raw_text.find(question)
    if i < 0:
        return "(question not found in corpus)"
    j = raw_text.find("\nQuestion:", i)
    return raw_text[i: j if j >= 0 else len(raw_text)]

print("\nGround Truth (from input.txt):")
print(get_ground_truth(test_prompt))


Loading model matrices from local memory space...


Loading weights:   0%|          | 0/39 [00:00<?, ?it/s]


--- Prompting Model with: 'Question: How do you register services in the .NET Dependency Injection container?' ---

🤖 Model Output:
Question: How do you register services in the. NET Dependency Injection container ? Answer You services the. file the. file the ' Services, between, between, between, use await to execution - until underlying completes
 ` ` csharp public Task()


){ public(;})
 ToList)

 activeUsers
 activeUsers
 ` `
 Question How you a query filter project from,,)

( =
: create class parameters
(; builder Services AddScoped)
.(; builder Services AddSingleton)
 app Run)
 app builder Services AddSingleton)
 app Run) $ Error{.(;} ` `

Ground Truth (from input.txt):
Question: How do you register services in the .NET Dependency Injection container?
Answer: You configure services inside the Program.cs file using the WebApplicationBuilder's Services collection, choosing between Transient, Scoped, or Singleton lifetimes.
```csharp
var builder = WebApplication.CreateBuilder(args)